In [1]:
from neo4j import GraphDatabase, RoutingControl

URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "new12345")


In [2]:
def add_friend(driver, name, friend_name):
    driver.execute_query(
        "MERGE (a:Person {name: $name}) "
        "MERGE (friend:Person {name: $friend_name}) "
        "MERGE (a)-[:KNOWS]->(friend)",
        name=name, friend_name=friend_name, database_="neo4j",
    )

In [3]:
def print_friends(driver, name):
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name "
        "RETURN friend.name ORDER BY friend.name",
        name=name, database_="neo4j", routing_=RoutingControl.READ,
    )
    for record in records:
        print(record["friend.name"])

In [4]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    print_friends(driver, "Arthur")

Guinevere
Lancelot
Merlin


In [5]:
import pandas as pd, os

df = pd.read_csv("data/events.csv")
df["dt"] = pd.to_datetime(df["timestamp"], unit="ms")
print(df.shape)
print(df["dt"].min(), "->", df["dt"].max())
print(df["event"].value_counts())

(2756101, 6)
2015-05-03 03:00:04.384000 -> 2015-09-18 02:59:47.788000
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64


In [ ]:
P1 = ("2015-06-03", "2015-08-03")   
P2 = ("2015-08-12", "2015-08-27")   

def cut(d, a, b):
    return d[(d.dt >= a) & (d.dt < pd.Timestamp(b) + pd.Timedelta(days=1))]

train, test = cut(df, *P1), cut(df, *P2)
print(len(train), len(test))

os.makedirs("data_p1", exist_ok=True)
train.drop(columns="dt").to_csv("data_p1/events.csv", index=False)
test.drop(columns="dt").to_csv("data/events_p2.csv", index=False)

1314193 288022


**PERTANYAAN PENDAHULUAN**

1. Apa nama database yang digunakan pada tutorial ini?
Database bernama neo4j, yaitu database default pada instance Neo4j. Terlihat dari parameter database_="neo4j" pada driver.execute_query(...) di fungsi add_friend dan print_friends.

2. Akses lain untuk mengakses database lain dalam 1 instance
Ada beberapa cara:
Python (driver): ganti nilai parameter, misalnya database_="namadb" pada execute_query, atau buka session khusus dengan driver.session(database="namadb").
Cypher (DDL/administrasi): SHOW DATABASES untuk melihat daftar database, CREATE DATABASE namadb untuk membuat database baru, dan DROP DATABASE namadb untuk menghapus.
Cypher (berpindah database di Query/Browser): :use namadb di Neo4j Browser, atau klausa USE namadb di awal query, misalnya USE namadb MATCH (n) RETURN n.
Catatan: 
Database system dipakai untuk perintah administrasi. Edisi Community umumnya hanya mendukung satu database pengguna (selain system), sedangkan multi-database penuh tersedia di Enterprise. Cek dengan SHOW DATABASES pada instance Anda.

3. Apa jenis graf pertemanan yang telah dibuat?
Graf berarah (directed graph), karena relasi (a)-[:KNOWS]->(friend) punya arah dari Arthur ke temannya. Karakteristik lainnya:
Tidak berbobot (relasi tidak punya properti bobot).
Homogen: satu jenis label simpul (Person) dan satu jenis relasi (KNOWS).
Berbentuk bintang (star): Arthur sebagai pusat dengan 3 relasi keluar ke Guinevere, Lancelot, dan Merlin (4 simpul, 3 relasi).
Karena arahnya satu sisi, Guinevere, Lancelot, dan Merlin tidak otomatis “mengenal” Arthur. Pertemanan ini asimetris.